# Exercise Sheet No. 8

---

> Machine Learning for Natural Sciences, Summer 2026, Prof. Pascal Friederich, pascal.friederich@kit.edu
>
> Tutor: jonas.teufel@kit.edu
>
> **Please ask questions in the forum/discussion board and only contact the Tutor when there are issues with the grading**
---

**Topic**: This exercise sheet will focus on recurrent neural networks such as GRUs and LSTMs. We'll be looking at the classification of ECG data and text sentiment classification.

Please add here your group members' names and student IDs. 

Names: 

IDs:

# 8 Recurrent Neural Networks


## 8.1 Working with Time-Series Data

**Normal Tabular Data.** With neural network models, we are usually working with relatively strictly defined data - at least regarding its data format. More specifically, when thinking about the data from the perspective of numeric arrays and matrices, we need to impose strict assumptions about the *shape* of this data. For a dense neural network that predicts tabular data, for example, the input vector always needs to have the same number and order of elements. For convolutional neural networks predicting based on image data, we need to make sure that the input images always have the same dimensions.

**Sequential Data.** For the class of data known as *sequential* data, strict assumptions about the shape are no longer possible for at least one of the data dimensions. One particularly illustrative example for sequential data is *text*. If we want to use neural networks to do classification on textual data, we run into the problem that text generally has different lengths. One often used example in the domain of text processing is the problem of [sentiment classification](https://en.wikipedia.org/wiki/Sentiment_analysis): In the most simple case, we could use a model to predict whether a given text has a positive (*"I really liked this book!"*) or a negative (*Worst movie I've ever seen!*) meaning. In a simplified manner we can encode text snippets as sequences of characters, where each character could be represented as a [one-hot encoded](https://en.wikipedia.org/wiki/One-hot) vector, for example.

In other domains, sequential data can be found whenever some measurements are recorded over periods of time. This, for example, includes measurements of seismic activities or climate data records that can later be used to perform analyses or predict forecasts of future observations. 

**Recurrent Neural Networks.** Due to the particular properties of sequential data, the special type of *recurrent neural networks (RNNs)* is required to handle the dynamic input shapes. These special network architectures treat the sequence dimension of the input data in a special manner to allow for different lengths. The common method of treating this dynamic sequence dimension is following: Instead of having dedicated parameters for the sequence dimension, the same existing network layers are applied for each time step independently. The network is able to maintain information about the whole sequence by accumulating in some kind of internal state representation that is shared between adjacent time steps.

In [ ]:
##### DO NOT CHANGE #####
#! pip show torch
import os
import io
import re
import sys
import time
import random
import hashlib
import zipfile
import tempfile
import typing as t
from collections import defaultdict
from copy import copy

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# render all plots on matplotlib's standard white background (also overrides a
# dark JupyterLab theme, which would otherwise make inline figures transparent)
plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.rnn as rnn
import torch.optim as optim

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
hashcheck = lambda v: hashlib.sha256(v.encode()).hexdigest()[:16]

def sigmoid(x: np.ndarray) -> np.ndarray:
    if isinstance(x, np.ndarray):
        x = x.astype(float)
        
    return 1.0 / (1.0 + np.exp(-x))

np.sigmoid = sigmoid
import urllib3.util.connection

# Force IPv4 for outbound downloads. bwsyncandshare.kit.edu's AAAA record
# points to an IPv6 endpoint that is unreachable from many networks (including
# bwJupyter); Python's HTTP clients don't implement Happy Eyeballs and would
# otherwise wait the full TCP timeout (~120s) before falling back to v4.
urllib3.util.connection.HAS_IPV6 = False



def nextcloud_download(url: str, raw: bool = False) -> str:
    """
    Downloads the content of a file from a nextcloud server and returns 
    it either as a string or a bytes object if the ``raw`` flag is set.
    """
    response = requests.get(f'{url}/download')
    content = response.content
    if not raw:
        content = content.decode('utf-8')
    
    return content



##### DO NOT CHANGE #####

**A Simple Example.** To get a basic understanding of how sequential tasks are structured, we will construct a simple sequential toy problem. We will consider a problem with two binary input values $u_t$, $v_t$ and one binary output value $y_t$. All of these values are signals over time and can be evaluated at different discrete time steps $t \in \{0, 1, 2, 3, \dots\}$. In this task, the current value of the output signal $y_t$ does not only depend on the current values of the input signals, but also on the input behavior that was observed in the previous time steps!

Specifically, the output function $y_t$ is defined to be $1$ only if the following two conditions are met:

- Across all previous time steps $\{ 0, \dots, t-1 \}$ a $1$ within the $u_t$ input signal was observed an *odd* number of times. (So only if there have been $1, 3, 5, 7, ...$ ones in the input signal previously)
- In the previous time step, the value of the $v_t$ input signal was $1$.

The previously described sequential input-output problem is illustrated in the following example:

In [ ]:
##### DO NOT CHANGE #####
ts = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15])
us = np.array([1, 0, 1, 0, 1, 1, 0, 1, 0, 1,  1,  0,  1,  0,  1,  1])
vs = np.array([0, 1, 0, 0, 0, 1, 0, 1, 0, 0,  1,  0,  1,  0,  1,  1])
ys = np.array([0, 0, 1, 0, 0, 0, 0, 0, 1, 0,  0,  1,  0,  0,  0,  1])

# For an odd number of vs, it displays how many us there were in the past

fig, (ax_us, ax_vs, ax_ys) = plt.subplots(
    ncols=1,
    nrows=3,
    figsize=(10, 15),
)

ax_us.step(ts-0.5, us, color='cyan')
ax_us.set_title('Input Signal u')
ax_us.set_xticks(ts)
ax_us.set_xlim([-1, 16])
ax_us.xaxis.grid(ls='--', alpha=0.5)

ax_vs.step(ts-0.5, vs, color='magenta')
ax_vs.set_title('Input Signal v')
ax_vs.set_xticks(ts)
ax_vs.set_xlim([-1, 16])
ax_vs.xaxis.grid(ls='--', alpha=0.5)

ax_ys.step(ts-0.5, ys, color='red')
ax_ys.set_title('Output Signal y')
ax_ys.set_xticks(ts)
ax_ys.set_xlim([-1, 16])
ax_ys.xaxis.grid(ls='--', alpha=0.5)
ax_ys.set_xlabel('time steps')

pass

##### DO NOT CHANGE #####

<div style="border: 1px solid #CEB037; border-radius: 3px; padding: 6px; background-color: #faf7e0ff; color: black;">
<strong>🛠️ Task 8.1 (2 points).</strong> In this task, implement the function <code>custom_step</code> which receives the momentary values of the two input signals <code>u</code> and <code>v</code> for each timestep. Additionally, the function receives a dictionary object <code>data</code> which is empty at the beginning, but retains its content between every time step! You may use this dictionary object to implement some kind of "memory" to solve the recurrent task. In each time step, the <code>custom_step</code> function is supposed to return the predicted output value <code>y</code> according to the previously described rule.
</div>

In [ ]:
def custom_step(u: float, v: float, data: dict) -> float:
    """
    This function receives the current value of the input signals ``u`` and ``v`` 
    as well as the memory storage ``data`` and is supposed to return the predicted value 
    which solves the previously described sequential toy problem.
    
    :param u: The current value of input signal u
    :param v: The current value of input signal v
    :param data: A dictionary object that retains its content between the time steps
    
    :returns: The predicted value for y
    """
    
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: test-8-1-custom-step - possible points: 2

data = defaultdict(int)

for t, u, v, y in zip(ts, us, vs, ys):
    y_pred = custom_step(u, v, data)
    print(f't: {t:02d} - true: {y} - pred: {y_pred}')
    assert np.isclose(y, y_pred, atol=0.1), f'predicted {y_pred} != expected {y}'
    
# NOTE: The hidden tests will perform the same kind of test with a number of randomly 
#       generated input and corresponding output signals of different lengths.


##### DO NOT CHANGE #####

## 8.2 Gated Recurrent Unit (GRU)

The first kind of RNN layer that we want to look at is the *Gated Recurrent Unit (GRU)*. The basic idea of this layer is that a hidden state vector $\mathbf{h}$ is maintained and updated throughout each time step of the sequence. These hidden state values act as a kind of "memory" that can store information about previous time steps. In a single forward pass, the GRU cell uses the current input vector $\mathbf{x}_t$ and the previous state vector $\mathbf{h}_{t-1}$ to compute the updated state vector $\mathbf{h}_t$ — and for the standard GRU this updated hidden state is *also* the output of the cell (there is no separate readout gate).

<img src="https://bwsyncandshare.kit.edu/s/89PNHisYaGMCJGW/download">

Concretely, the input and previous state are concatenated into $\mathbf{i} = (\mathbf{x}_t \,\|\, \mathbf{h}_{t-1})$ and the cell computes:

- The *reset* gate $\mathbf{r}_t = \sigma(W_r\,\mathbf{i} + \mathbf{b}_r)$ selectively determines which of the previous hidden state values may influence the new candidate state (each multiplied by a $[0, 1]$ factor).
- The *candidate* state $\tilde{\mathbf{h}}_t = \tanh\!\big(W_h\,(\mathbf{x}_t \,\|\, \mathbf{r}_t \odot \mathbf{h}_{t-1}) + \mathbf{b}_h\big)$ proposes new state values from the input and the reset previous state. Because of the $\tanh$, these values always lie in $[-1, 1]$.
- The *update* gate $\mathbf{z}_t = \sigma(W_z\,\mathbf{i} + \mathbf{b}_z)$ determines, for each state value, whether to keep the previous value or use the candidate, via the weighted average $\mathbf{h}_t = (1 - \mathbf{z}_t)\odot\mathbf{h}_{t-1} + \mathbf{z}_t\odot\tilde{\mathbf{h}}_t$.

<div style="border: 1px solid #CEB037; border-radius: 3px; padding: 6px; background-color: #faf7e0ff; color: black;">
<strong>🛠️ Task 8.2 (1 point).</strong> To familiarize yourself with the architecture of the GRU cell, your next task is simply to determine the appropriate shape tuples for the weight matrices $W_r, W_z, W_h$ given the fixed shapes of the input signal <code>x_shape</code> and the hidden vector <code>h_shape</code>.
</div>

In [ ]:

x_shape = (5, )
h_shape = (7, )

# TASK: Given the example shapes for the input and hidden vectors, fill in the
#       expected shapes of the various weight matrices that are used in the GRU architecture.

W_r_shape: tuple[int, int] = None
W_z_shape: tuple[int, int] = None
W_h_shape: tuple[int, int] = None

# YOUR CODE HERE
raise NotImplementedError()


In [ ]:
##### DO NOT CHANGE #####
# ID: test-8-2-gru-shapes - possible points: 1

assert isinstance(W_r_shape, tuple)
assert len(W_r_shape) == 2
assert hashcheck(f'W_r{sum(W_r_shape)}') == 'f1e3f6eba02f53b1', 'W_r shape likely incorrect'

assert isinstance(W_z_shape, tuple)
assert len(W_z_shape) == 2
assert hashcheck(f'W_z{sum(W_z_shape)}') == 'e5f3c32506daf289', 'W_z shape likely incorrect'

assert isinstance(W_h_shape, tuple)
assert len(W_h_shape) == 2
assert hashcheck(f'W_h{sum(W_h_shape)}') == '18645d6bfc1fc4b4', 'W_h shape likely incorrect'

# NOTE: The hidden tests will simply check for the exact expected shape.


##### DO NOT CHANGE #####

<div style="border: 1px solid #CEB037; border-radius: 3px; padding: 6px; background-color: #faf7e0ff; color: black;">
<strong>🛠️ Task 8.3 (4 points).</strong> Complete the implementation of the <code>GRU</code> class below. First, in the constructor, initialize the weight matrices and bias vectors <code>W_r, b_r, W_z, b_z, W_h, b_h</code> with the correct shapes (initialize every entry to zero, just like in the LSTM task that follows). Then implement the <code>forward</code> method, which takes the current input vector $\mathbf{x}_t$ and the previous hidden state $\mathbf{h}_{t-1}$ and returns the updated hidden state $\mathbf{h}_t$, following the equations above.
</div>

In [ ]:
class GRU:
    """numpy implementation of the (standard) GRU cell."""

    def __init__(self,
                 num_inputs: int,
                 num_states: int,
                 ):
        self.num_inputs = num_inputs
        self.num_states = num_states

        # reset gate
        self.W_r = None; self.b_r = None
        # update gate
        self.W_z = None; self.b_z = None
        # candidate state
        self.W_h = None; self.b_h = None

        # YOUR CODE HERE
        raise NotImplementedError()

    def forward(self,
                x: np.ndarray,
                h: np.ndarray,
                ) -> np.ndarray:
        """Given the current input ``x`` (shape (num_inputs,)) and the previous hidden
        state ``h`` (shape (num_states,)), return the updated hidden state ``h_new``."""
        # YOUR CODE HERE
        raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: test-8-3-gru-constructor - possible points: 1

# checking one example instantiation of the GRU class
gru = GRU(2, 3)
assert isinstance(gru, GRU)

# all matrices and bias vectors have to be numpy arrays
for name in ['W_r', 'b_r', 'W_z', 'b_z', 'W_h', 'b_h']:
    assert isinstance(getattr(gru, name), np.ndarray), f'{name} must be a numpy array'

# all 'W' variables are matrices (2-dim), all 'b' variables are vectors (1-dim)
for w in ['W_r', 'W_z', 'W_h']:
    assert getattr(gru, w).ndim == 2, f'{w} must be 2-dimensional'
for b in ['b_r', 'b_z', 'b_h']:
    assert getattr(gru, b).ndim == 1, f'{b} must be 1-dimensional'

# checking the exact shapes for this example (inputs=2, states=3)
assert gru.W_r.shape == (3, 5) and gru.b_r.shape == (3,)

# NOTE: the hidden tests construct GRUs with random input/state dimensions
#       and check the shapes of all weight and bias arrays.


##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ID: test-8-3-gru-forward - possible points: 3

# checking the forward pass for one example (all-zero weights from the constructor)
gru = GRU(2, 3)
x = np.array([1.0, 1.0])
h = np.array([2.0, 3.0, -2.0])
h_new = gru.forward(x, h)

assert isinstance(h_new, np.ndarray) and h_new.shape == (3,)

# with all-zero weights: r = z = 0.5, h_tilde = 0  ->  h_new = 0.5 * h
assert np.allclose(h_new, [1.0, 1.5, -1.0], rtol=0.1), 'h_new calculation incorrect'

# NOTE: the hidden tests compare the forward pass against a reference implementation
#       for random weights and inputs.


##### DO NOT CHANGE #####

## 8.3 Long-Short Term Memory (LSTM)

The *Long-Short Term Memory (LSTM)* is an alternative to the GRU cell which was introduced by [Hochreiter and Schmidhuber](https://doi.org/10.1162/neco.1997.9.8.1735). The following image illustrates the basic architecture of an LSTM cell:

<img src="https://bwsyncandshare.kit.edu/s/NbKxq9CeMjCxZCg/download">

Much like the GRU cell, the LSTM cell also internally maintains a hidden representation $\mathbf{h}_t$ which acts as a kind of memory that can be used to keep track of an internal *state* across different time steps. However, unlike the GRU cell, the LSTM cell naturally includes a *readout gate* which separates the internal hidden state $\mathbf{h}_t$ and the readout signal $\mathbf{p}_t$. Additionally, the selection mechanism that determines which parts of the previous state $\mathbf{h}_{t-1}$ to keep and which parts of the new state $\mathbf{\tilde{h}}_{t}$ to update is implemented differently.

Specifically, the LSTM cell can be described by the following equations. The two selection signals $f_t, i_t$ determine how to compose the current time step's hidden state vector from the previous and updated version, where 

$$
f_t = \sigma(W_f \cdot (x_t || h_{t-1}) + b_f)
$$

is the selection signal that determines which parts of the previous time step's hidden vector to keep and 

$$
i_t = \sigma(W_i \cdot (x_t || h_{t-1}) + b_i)
$$

is the selection signal that determines which parts of the updated state vector to use. The updated state vector 

$$
\tilde{h}_t = \mathrm{tanh}(W_h \cdot (x_t || h_{t-1}) + b_h)
$$

is calculated by a linear layer transformation based on the input vector and the previous state vector.
Finally, the new hidden state vector

$$
h_t = f_t \cdot h_{t-1} + i_t \cdot \tilde{h}_t
$$

is a linear combination of the previous state vector and the updated state vector.
The readout signal 

$$
p_t = \mathrm{tanh}(h_t) \cdot \sigma(W_o \cdot (x_t || h_{t-1}) + b_o)
$$

is derived from a masking of the current time step's hidden state vector.

<div style="border: 1px solid #CEB037; border-radius: 3px; padding: 6px; background-color: #faf7e0ff; color: black;">
<strong>🛠️ Task 8.4 (5 points).</strong> The goal of this task is to complete the implementation of the <code>LSTM</code> class below. To do this, you'll first have to properly initialize the weight matrices and bias vectors $W_f, b_f, W_i, b_i, W_h, b_h, W_o, b_o$ in the constructor of the class. In this task you can simply initialize all weight and bias arrays with all values set to zero. Additionally you'll have to implement the <code>forward</code> method which takes the current input vector $\mathbf{x}_t$ and the previous hidden state vector $\mathbf{h}_{t-1}$ as inputs and is expected to return a tuple consisting of the updated hidden state vector $\mathbf{h}_t$ and the readout vector $\mathbf{p}_t$.
</div>

In [ ]:
class LSTM:
    """
    numpy implementation of the Long-Short Term Memory (LSTM) cell.
    """
    
    def __init__(self,
                 num_inputs: int,
                 num_states: int,
                 ):
        
        self.num_inputs = num_inputs
        self.num_states = num_states
        
        # selection gate
        self.W_f: np.ndarray = None
        self.b_f: np.ndarray = None
            
        # selection gate
        self.W_i: np.ndarray = None
        self.b_i: np.ndarray = None
            
        # update gate
        self.W_h: np.ndarray = None
        self.b_h: np.ndarray = None
        
        # readout gate
        self.W_o: np.ndarray = None
        self.b_o: np.ndarray = None
            
        # YOUR CODE HERE
        raise NotImplementedError()
    
    def forward(self,
                x: np.ndarray,
                h: np.ndarray,
                ) -> tuple[np.ndarray, np.ndarray]:
        """
        Given the current time step's input array ``x`` of the shape (input_dim, ) and the 
        previous time step's hidden state array ``h`` of the shape (hidden_dim, ), this 
        method returns a tuple (``h_new``, ``p``) consisting of the updated hidden state 
        vector and the readout vector of the shape (hidden_dim, ).
        """
        
        # YOUR CODE HERE
        raise NotImplementedError()
        


In [ ]:
##### DO NOT CHANGE #####
# ID: test-8-4-lstm-constructor - possible points: 1

# checking for one example instantiation of the LSTM class
lstm = LSTM(2, 3)
assert isinstance(lstm, LSTM)

# all matrices and bias vectors have to be numpy arrays!
assert isinstance(lstm.W_f, np.ndarray)
assert isinstance(lstm.b_f, np.ndarray)
assert isinstance(lstm.W_i, np.ndarray)
assert isinstance(lstm.b_i, np.ndarray)
assert isinstance(lstm.W_h, np.ndarray)
assert isinstance(lstm.b_h, np.ndarray)
assert isinstance(lstm.W_o, np.ndarray)
assert isinstance(lstm.b_o, np.ndarray)

# all "W" variables have to be matrices (2-dimensional)
assert len(lstm.W_f.shape) == 2
assert len(lstm.W_i.shape) == 2
assert len(lstm.W_h.shape) == 2
assert len(lstm.W_o.shape) == 2

# all "b" variables have to be biases (1-dimensional)
assert len(lstm.b_f.shape) == 1
assert len(lstm.b_i.shape) == 1
assert len(lstm.b_h.shape) == 1
assert len(lstm.b_o.shape) == 1

# checking the exact shape of one gate as an example
assert hashcheck(f'W_f{lstm.W_f.shape}') == '9e7dcaf998a20515', 'W_f likely wrong shape'
assert hashcheck(f'b_f{lstm.b_f.shape}') == 'd7ff6f69428e93c9', 'b_f likely wrong shape'

# NOTE: The hidden tests will construct a number of test cases with combinations of 
#       random input and hidden dimensions. For each combination, a new LSTM instance 
#       will be created and the shapes of the weights and bias arrays will be checked 
#       against the expected shapes.


##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ID: test-8-4-lstm-forward - possible points: 4

lstm = LSTM(2, 3)
x = np.array([1.0, 1.0])
h = np.array([2.0, 3.0, -2.0])

h_new, p = lstm.forward(x, h)

assert isinstance(h_new, np.ndarray)
assert h_new.shape == (3, )

assert isinstance(p, np.ndarray)
assert p.shape == (3, )

# For the given input arrays these are the expected  outputs of the forward method
# HINT: For these results, the assumption is that all the matrices and biases are 
#       initialized with all zero values in the constructor of the LSTM class
assert np.isclose(h_new, np.array([1.0, 1.5, -1.0]), rtol=0.1).all(), 'h_new calculation incorrect'
assert np.isclose(p, np.array([ 0.380,  0.452, -0.380]), rtol=0.1).all(), 'p calculation incorrect'

# NOTE: The hidden tests will execute the forward pass for a few sample input and hidden 
#       vectors and compare the results of the given implementation with the ground truth 
#       implementation.


##### DO NOT CHANGE #####

### Can they *count*?

The above-introduced GRU and LSTM cell look similar but they handle memory very differently. In the following section we want to explore this difference by looking at a simple *counting* task: The input of the memory cells will be a series of pulses and the goal will be that the output of the memory cells tracks how many pulses have occurred up to that point in time. One could argue that such a counting task is one of the most basic flavors of "memory".

**📊 Visualization.** The visualization below shows the input pulses in blue, the "ground truth" pulse count as a dotted grey line and the corresponding cell states in red and orange.

In [ ]:
##### DO NOT CHANGE #####
import matplotlib.pyplot as plt

# a train of input pulses (1 = a pulse arrived, 0 = nothing)
pulses = np.array([0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1], dtype=float)
true_count = np.cumsum(pulses)            # the running number of pulses seen so far
ts = np.arange(len(pulses))

# an LSTM configured as an integrator: forget gate open (f~1, keep the past),
# input gate opens on a pulse (i~1 when pulse=1), candidate ~ +1
lstm_counter = LSTM(1, 1)
lstm_counter.W_f = np.array([[0.0, 0.0]]); lstm_counter.b_f = np.array([10.0])
lstm_counter.W_i = np.array([[10.0, 0.0]]); lstm_counter.b_i = np.array([-5.0])
lstm_counter.W_h = np.array([[0.0, 0.0]]); lstm_counter.b_h = np.array([10.0])
lstm_counter.W_o = np.array([[0.0, 0.0]]); lstm_counter.b_o = np.array([0.0])

# a GRU: update gate opens on a pulse, candidate ~ +1.
gru_counter = GRU(1, 1)
gru_counter.W_r = np.array([[0.0, 0.0]]); gru_counter.b_r = np.array([10.0])
gru_counter.W_z = np.array([[10.0, 0.0]]); gru_counter.b_z = np.array([-5.0])
gru_counter.W_h = np.array([[0.0, 0.0]]); gru_counter.b_h = np.array([10.0])

def run_counter(cell):
    """Feed the pulse train through ``cell`` one time step at a time and record
    the hidden-state value after each step.

    This works for *both* of our cells even though their ``forward`` returns
    differ: ``GRU.forward(x, h)`` returns just the new hidden state ``h_t``,
    while ``LSTM.forward(x, h)`` returns an ``(h_t, readout)`` tuple. We only
    care about ``h_t`` here, so we unpack the tuple whenever there is one.
    """
    h = np.zeros(1)   # hidden state, starts at zero (no pulses seen yet)
    states = []       # the hidden-state value recorded after each time step

    for pulse in pulses:
        x = np.array([pulse])               # the cell expects a length-1 input vector
        out = cell.forward(x, h)            # advance the cell by one time step
        h = out[0] if isinstance(out, tuple) else out  # LSTM -> (h_t, readout); keep h_t
        states.append(float(h[0]))         # the state is a 1-element vector -> scalar

    return np.array(states)

# Run each pre-configured cell over the same pulse train and collect its states.
lstm_states = run_counter(lstm_counter)
gru_states = run_counter(gru_counter)

fig, (ax_l, ax_g) = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax in (ax_l, ax_g):
    ax.bar(ts, pulses, width=0.15, color='tab:cyan', alpha=0.6, label='pulse')
    ax.step(ts, true_count, where='mid', ls='--', color='gray', label='true count')
    ax.set_xlabel('time step'); ax.set_xticks(ts); ax.grid(alpha=0.3)
ax_l.plot(ts, lstm_states, 'o-', color='tab:orange', label='LSTM state')
ax_l.set_title('LSTM'); ax_l.set_ylabel('value')
ax_l.legend(loc='upper left')
ax_g.plot(ts, gru_states, 'o-', color='tab:red', label='GRU state')
ax_g.set_title('GRU')
ax_g.legend(loc='upper left')
plt.tight_layout(); plt.show()

##### DO NOT CHANGE #####

<div style="border: 1px solid #CEB037; border-radius: 3px; padding: 6px; background-color: #faf7e0ff; color: black;">
<strong>🛠️ Task 8.5 (1 point).</strong> Based on the demonstration above, consider the following four statements about the counting behavior of the GRU and LSTM cells. Decide which of them are <em>true</em> and assign the <strong>list of the (1-based) numbers</strong> of all true statements to the <code>answer</code> variable below — for example, <code>answer = [1, 3]</code> if you believe statements 1 and 3 are true.
<ol>
<li style="font-weight: bold;"><span style="font-weight: normal;">With the right weights and biases, the GRU could be configured to count the pulses the way the LSTM does.</span></li>
<li style="font-weight: bold;"><span style="font-weight: normal;">The LSTM's cell-state update is additive, which is what lets its value climb past 1 and track the running total.</span></li>
<li style="font-weight: bold;"><span style="font-weight: normal;">The real reason the LSTM can count is that it has an extra readout (output) gate that the GRU lacks.</span></li>
<li style="font-weight: bold;"><span style="font-weight: normal;">A key reason the LSTM can count is that its forget gate stays open: if the forget gate instead closed whenever a pulse arrived, the cell would overwrite its memory instead of adding to the running total.</span></li>
</ol>
</div>

In [ ]:
# TASK: Assign the list of the 1-based numbers of all statements you believe to
#       be true to the `answer` variable below. For example: answer = [1, 3].

answer: list[int] = None

# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: test-8-5 - possible points: 1

# A quick check on the *format* of your answer (this does not check correctness).
assert isinstance(answer, list), 'answer must be a list, e.g. answer = [1, 3]'
assert all(isinstance(i, int) for i in answer), 'answer must contain integers'
assert all(1 <= i <= 4 for i in answer), 'the statement numbers must be between 1 and 4'

# NOTE: The hidden test below checks that your selection is EXACTLY correct -- you
#       only receive the point if the set of selected statements matches perfectly.


##### DO NOT CHANGE #####

## 8.4 ECG Classification

**ECG Classification.** In this section, we want to apply the problem to an actual classification task on sequentially structured data. Specifically, we are going to use a dataset of [*Electrocardiogram (ECG)*](https://en.wikipedia.org/wiki/Electrocardiography) data. An ECG is a medical test that measures the electrical activity of the heart over a period of time using electrodes placed on the skin. This data is inherently sequential and is characterized by temporal patterns that are crucial for identifying heart conditions. The resulting measurement is a function over time and may, for example, look similar to the following image:

<img src="https://upload.wikimedia.org/wikipedia/commons/a/a2/12_lead_ECG_of_a_26_year_old_male.jpg" width="80%">

Such ECG measurements can be used to detect certain heart conditions which will create certain characteristic deviations in the course of the signal. To diagnose these heart conditions, specifically trained medical staff is able to identify the characteristic deviations in the signal. Alternatively, since the deviations caused by the various diseases follow a characteristic pattern, it is also possible to apply a machine learning model to use a patient's ECG signal as an input to classify a possible heart condition.

**ECG Dataset.** In this section we'll be using a dataset of ECG data that consists of roughly 5k patients' records. Each record is a function over time representing only a single heartbeat within a larger ECG measurement. The records of different patients have different lengths.

**Data Loading.** The first step in working with the given ECG dataset is to load the dataset into memory. However, compared to previous datasets you've worked with, sequential data is a less convenient data format. Since the different elements of a sequential dataset may have different sequence lengths, they cannot be easily encoded into a single CSV file. Because of this, sequential datasets are sometimes given in more *unusual* data formats, where the individual sequences are often encoded as individual files. This circumstance can somewhat complicate the process of loading sequential datasets into a sensible format for the subsequent training of the machine learning models.

The given ECG dataset is available as a ZIP archive ``ecg.zip``. This archive contains multiple files - exactly *one* for each element in the dataset. Each of these files is a ``.npy`` numpy file, which contains the binary encoding of a numpy array. The content of these files encodes a 1-dimensional numpy array that represents the ECG signal of one patient. The *names* of the files contain the information about the patient ID from which the sample was recorded, as well as the target label (normal/abnormal) associated with each recording in the following format:

``"patient_{patient_id}_{label}.npy"``

Some examples are: 

- ``patient_0001_normal.npy``
- ``patient_0022_normal.npy``
- ``patient_2906_abnormal.npy``

<div style="border: 1px solid #CEB037; border-radius: 3px; padding: 6px; background-color: #faf7e0ff; color: black;">
<strong>🛠️ Task 8.6 (2 points).</strong> In this task, the goal is to implement the <code>load_dataset</code> function. This function accepts the path to the dataset ZIP archive and is supposed to return a list of dictionaries, where each dict represents a single element in the dataset. Each of these dicts should contain the following three key value pairs:
<ul>
<li><code>id</code>: The <em>integer</em> ID of the patient from whom the ECG recording was collected.</li>
<li><code>target</code>: The <em>integer</em> target value annotation, which is 0 for ECG data labeled as "normal" and is 1 for "abnormal" ECG data</li>
<li><code>signal</code>: A <em>list</em> of <em>lists</em> containing <em>float</em> values that represents the time series recording of the ECG itself.</li>
</ul>
The path to the downloaded dataset archive is available in the <code>zip_path</code> variable.
</div>

An example for the expected final result may look like this:

```python
dataset = [
    {
        'id': 0,
        'target': 0,
        # Wrapping each element in a list may seem redundant but we do this here because
        # the model later on expects each sequence to be a 2-dimensional tensor!
        'signal': [[0.9], [0.8], ...],
    },
    {
        'id': 1,
        'target': 0,
        'signal': [[0.7], [0.8], ...],
    },
    ...
]
```

In [ ]:
##### DO NOT CHANGE #####
# The dataset archive ZIP file will be downloaded and stored on the local system at 
# the path given in "zip_path"
content = nextcloud_download('https://bwsyncandshare.kit.edu/s/xMctqCzFp3XRtKM', raw=True)
zip_path = 'ecg_.zip'
with open(zip_path, 'wb') as file:
    file.write(content)

##### DO NOT CHANGE #####

In [ ]:
# TASK: Implement the following function which loads the dataset from its ZIP 
#       archive format into a list of dictionaries that contain all the relevant 
#       information about the various data points.

# HINT: If not already the case, it makes sense to familiarize yourself with the 
#       ".npy" numpy file format and how to load the data contained in these kinds 
#       of files.

def load_dataset(path: str) -> list[dict]:
    """
    This function loads the dataset from the ZIP archive at the given absolute string 
    ``path`` and returns a list of dict objects, where each dict contains the information 
    about one element of the dataset.
    
    Each dict contains the following 3 keys:
    - id: The unique integer id of the element
    - target: The integer target value annotation
    - signal: A list of list float values that represents the signal
    """
    
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
##### DO NOT CHANGE #####
# ID: test-8-6-load-dataset - possible points: 2

# making sure the dataset ZIP archive exists
assert os.path.exists(zip_path)
assert os.path.isfile(zip_path)
assert zip_path.endswith('.zip')

# loading the dataset
dataset = load_dataset(zip_path)
assert len(dataset) == 4766

# simply checking for the correctness of the first element as a proxy for 
# the entire dataset.
first_element = dataset[0]
assert 'id' in first_element
assert isinstance(first_element['id'], int)

assert 'target' in first_element
assert isinstance(first_element['target'], int)

assert 'signal' in first_element
assert isinstance(first_element['signal'], list)
assert len(first_element['signal']) > 0
assert len(np.array(first_element['signal']).shape) == 2, "The signal should be 2-dim arrays!"

# NOTE: The hidden tests will perform similar type and shape checks for all the 
#       elements of the dataset.


##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
target_data_map: dict[int, list] = defaultdict(list)
for data in dataset:
    target_data_map[data['target']].append(data['signal'])

# Based on this distinction of normal(0) and abnormal(1) we can now plot some 
# example signals for both labels to see if there is any distinguishable visual 
# difference between them.

fig, (ax_normal, ax_abnormal) = plt.subplots(
    ncols=1,
    nrows=2,
    figsize=(10, 15)
)

num_samples = 20
for i in range(num_samples):
    ax_normal.plot(target_data_map[0][i], color='blue', alpha=0.2)
    ax_abnormal.plot(target_data_map[1][i], color='red', alpha=0.2)

ax_normal.set_title('Normal ECG Signals')
ax_normal.set_xlabel('time')
ax_normal.set_ylabel('signal value')

ax_abnormal.set_title('Abnormal ECG Signals')
ax_abnormal.set_xlabel('time')
ax_abnormal.set_ylabel('signal value')

##### DO NOT CHANGE #####

As we can see in these plots, the normal and abnormal ECG signals show quite a distinctive visual difference! While the normal signals are relatively constant in the middle section, the abnormal signals show a local minimum and maximum in the middle.

This obvious visual difference already indicates that the RNN model will likely also be able to achieve good prediction performance.

**PyTorch Sequential Data Handling.** One can imagine that working with sequences of different lengths also introduces challenges on a technical level. Deep Learning frameworks such as PyTorch handle data in the form of *tensors*. During the training of the model, each individual element can be considered and various elements can be combined into a *batch* tensor of higher-dimension to increase the computational efficiency of the training process, as it allows the model to train with multiple samples at the same time. This batching of multiple elements is relatively easy with static data types such as tabular data or image data. In the case of image data, for example, every image can be represented as a matrix of the same dimensions and multiple images can easily be combined by stacking them in a third dimension. However, for sequential data - where elements can have different dimensions - the batching process isn't as easy. Trying to stack sequences of different lengths is like trying to construct a matrix with column vectors of different lengths - which is simply not possible according to the definition of a matrix.
 
One possible solution to the problem of sequence batching is to simply not use batching. If the model is trained by only ever seeing a single input sequence at a time, the issue of different sequence lengths never arises. While this is certainly possible, it greatly decreases the computational efficiency of the training process and might have a negative impact on model performance as well.

Contrary to just using single sequences, PyTorch offers the option to use batching for sequential data by using *padding*. In any given batch of sequences, all sequences are artificially extended to the length of the longest sequence by appending arbitrary elements to the end (usually zeros). Additionally, for every element in the batch we store the information about the length of the original sequence. The combination of this information - padded sequence tensor and sequence lengths - is then given to the model to perform the predictions and subsequent model updates.

<img src="https://juditacs.github.io/assets/padded_sequence.png">

<div style="border: 1px solid #CEB037; border-radius: 3px; padding: 6px; background-color: #faf7e0ff; color: black;">
<strong>🛠️ Task 8.7 (2 points).</strong> The goal of this task is to implement the <code>batch_sequences</code> function, which accepts a list <code>seq_list</code> of sequential dataset elements and returns a tuple <code>(batch_tensor, lengths)</code>, where <code>batch_tensor</code> is a <code>torch.Tensor</code> instance which contains all of these sequences at the same time and which can be used later on to train the RNN models in a batched manner. <code>lengths</code> is supposed to be a list containing the integer length values of the input sequences.
<br><br>
For this task we assume that there are <code>num_sequences</code> sequences given in the input list. Each element of that list is itself a list. These lists are the actual sequential dataset elements and may have different lengths, but we assume that there exists one list which is the longest with a length of <code>max_length</code>. Each element of these lists is again a vector (list) with the number of <code>num_dimensions</code>. The function should return a float pytorch tensor object with the shape:
<br><br>
<code>(num_sequences, max_length, num_dimensions)</code>
<br><br>
To do this, the function should perform <em>zero-padding</em> for the sequences of different lengths.
</div>

The following example illustrates the expected behavior of the batching process:

```python
# input:
seq_list = [
    [[1, 1], [2, 2], [3, 3]],
    [[4, 4]],
]

# output:
batch_tensor = torch.Tensor([
    [[1.0, 1.0], [2.0, 2.0], [3.0, 3.0]],
    [[4.0, 4.0], [0.0, 0.0], [0.0, 0.0]],
]) # shape: (2, 3, 2)
lengths = [
    3,
    1,
]
```

In [ ]:
import torch
import torch.nn.utils.rnn as rnn

# TASK: Implement the batch_sequences function which turns the raw list of sequences 
#       into a single batched torch tensor.

# HINT: torch may already provide a suitable utility function that solves the 
#       majority of this task for you.

# HINT: Make sure that the tensor has the data type torch.float32!

def batch_sequences(seq_list: list[list]) -> tuple[torch.Tensor, list[int]]:
    """
    Given a list of vector sequences ``seq_list`` this function performs zero padding 
    and stacking of the sequences to return a single batched torch tensor with the 
    shape (num_sequences, max_length, num_dimensions)
    """
    
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: test-8-7-batch-sequences - possible points: 2

seq_list_1 = [
    [[1.0], [2.0], [3.0]],
    [[1.0], [1.0]],
    [[3.0], [9.0], [8.0], [3.5]]
]
batch_1, lengths_1 = batch_sequences(seq_list_1)
assert isinstance(batch_1, torch.Tensor)
assert batch_1.shape == (3, 4, 1)
assert tuple(lengths_1) == (3, 2, 4)

seq_list_2 = [
    [[1.0, 1.0], [2.0, 2.0]],
    [[1.0, 0.0]],
    [[1.0, 1.0], [2.0, 1.0], [2.0, 0.0]],
    [[2.0, 2.0]],
]
batch_2, lengths_2 = batch_sequences(seq_list_2)
assert isinstance(batch_2, torch.Tensor)
assert batch_2.shape == (4, 3, 2)
assert tuple(lengths_2) == (2, 1, 3, 1)

# NOTE: The hidden tests will construct some randomly generated test cases, where 
#       the different input sequences are randomly generated and the derived batch 
#       tensors are checked for their shape.


##### DO NOT CHANGE #####

**PyTorch Recurrent Neural Networks.** In the previous sections, we've worked with custom implementations of the ``GRU`` and ``LSTM`` recurrent neural network cells. However, deep learning frameworks usually provide default implementations for well-known architectures like this. In PyTorch, these default implementations can easily be accessed via the ``torch.nn.GRU`` and ``torch.nn.LSTM`` classes.

From a technical perspective, the only peculiarity when working with recurrent cells like these is how to properly handle the padded input batches. As previously discussed, the input to the model's forward pass is the padded batch of sequences ``x`` and the list of original sequence ``lengths``. However, the actual recurrent layers expect the data in the *"packed"* format. The PyTorch utility function ``pack_padded_sequence`` can be used to convert the padded format into the packed format.

<div style="border: 1px solid #CEB037; border-radius: 3px; padding: 6px; background-color: #faf7e0ff; color: black;">
<strong>🛠️ Task 8.8 (5 points).</strong> In this section, your task is to complete the implementations of the <code>GruModel</code> and <code>LstmModel</code> classes. Specifically, you'll have to implement the models constructor in the <code>__init__</code> method and the model forward pass in the <code>forward</code> method.
<br><br>
Each model should perform a binary classification for <em>each sequence</em> in the given batch of input data. This means for each sequence in the input batch the model should return a single raw <em>logit</em> value (an unbounded real number) where lower values represent the "normal" class (0) and higher values represent the "abnormal" class (1). We deliberately do <em>not</em> apply a sigmoid inside the model: the numerically stable <a href="https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html">binary cross entropy with logits loss</a> applies the sigmoid internally during training, and a <code>torch.sigmoid</code> is applied only at inference time to recover a probability in $[0, 1]$.
<br><br>
To achieve this you'll have to extend the given model classes with additional transformation layers such that the forward method outputs a tensor of the shape <code>(batch_size, 1)</code> of raw logits.
</div>

In [ ]:

class GruModel(nn.Module):
    """
    Implementation for a model that performs time-series classification based
    on the *gated recurrent unit (GRU)* cell architecture.
    """

    def __init__(self,
                 input_dim: int,
                 hidden_dim: int,
                 ):
        nn.Module.__init__(self)
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

        self.lay_gru = nn.GRU(input_dim, hidden_dim, batch_first=True)

        # YOUR CODE HERE
        raise NotImplementedError()

    def forward(self, x: torch.Tensor, lengths: list[int]) -> torch.Tensor:
        """
        Receives the padded input tensor ``x`` of shape (batch_size, max_length, input_dim) and the
        list ``lengths`` of the batch sequence lengths as input and returns a prediction output
        tensor of the shape (batch_size, 1) which consists of the raw classification logits
        for each sequence in the batch.
        """
        packed_input = rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        packed_output, hidden = self.lay_gru(packed_input)

        # YOUR CODE HERE
        raise NotImplementedError()


class LstmModel(nn.Module):
    """
    Implementation of a model that performs time-series classification based
    on the *long-short term memory* cell architecture.
    """

    def __init__(self,
                 input_dim: int,
                 hidden_dim: int,
                 ):
        nn.Module.__init__(self)
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim

        self.lay_lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)

        # YOUR CODE HERE
        raise NotImplementedError()

    def forward(self, x: torch.Tensor, lengths: list[int]) -> torch.Tensor:
        """
        Receives the padded input tensor ``x`` of shape (batch_size, max_length, input_dim) and the
        list ``lengths`` of the batch sequence lengths as input and returns a prediction output
        tensor of the shape (batch_size, 1) which consists of the raw classification logits
        for each sequence in the batch.
        """
        packed_input = rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        packed_output, (hidden, cell) = self.lay_lstm(packed_input)

        # YOUR CODE HERE
        raise NotImplementedError()

**Aggregating a sequence into a single prediction.** A recurrent layer returns one output vector *per time step*, but for classification we need to collapse this variable-length sequence into a single fixed-size vector per sample. There are several common strategies for this *aggregation* step:

- **Last hidden state** (used in the skeleton above): take `hidden[-1]`, the hidden state after the last *real* (non-padded) time step. Because the input is packed with `pack_padded_sequence`, the returned `h_n` already corresponds to each sequence's true final step, so this automatically ignores the padding. This is the standard, still-recommended approach for a single-layer, unidirectional RNN.
- **Mean / max pooling over time**: unpack the outputs with `pad_packed_sequence`, then average (or take the element-wise maximum) over the valid time steps, using `lengths` to mask out the padding. Pooling draws on the whole sequence instead of only the final step and often classifies at least as well.
- **Attention pooling**: learn a scalar score per time step, mask the padded positions to $-\infty$, `softmax` over the valid steps and take the weighted average. This lets the model decide which time steps matter most.

> ⚠️ **Caveat for bidirectional / multi-layer RNNs.** `hidden[-1]` only unambiguously means "the last time step" for a *single-direction* RNN. For a bidirectional or multi-layer RNN, `h_n` has shape `(num_layers * num_directions, batch, hidden)` and `hidden[-1]` silently selects just one layer/direction — a common source of bugs. In those cases concatenate the relevant slices (e.g. the forward and backward final states) or use pooling instead.

In [ ]:
##### DO NOT CHANGE #####
# ID: test-8-8-gru-model - possible points: 1

input_dim = 3
hidden_dim = 64
model_gru = GruModel(input_dim, hidden_dim)

# At first we check if there even are any GRU modules used in the model
assert any([isinstance(child, nn.GRU) for child in model_gru.children()]), (
    'The GruModel class likely does not implement torch GRU layer'
)
# Then we make sure that the hidden size for those GRU modules has been 
# set correctly.
for child in model_gru.children():
    if isinstance(child, nn.GRU):
        assert child.hidden_size == hidden_dim, 'GRU layer hidden size incorrect'

# The general idea of the functional tests here is that we create random input 
# vectors, perform a model forward pass and see if the model output has the correct 
# shape, consists of finite logit values, and whether or not it is possible to compute a 
# proper backward pass / gradient calculation for the given model implementation.
for _ in range(3):
    
    batch_size = np.random.randint(2, 10)
    max_length = np.random.randint(10, 20)
    
    x = torch.tensor(
        np.random.rand(batch_size, max_length, input_dim),
        dtype=torch.float32,
        requires_grad=True,
    )
    lengths = [random.randint(2, max_length) for _ in range(batch_size)]
    
    # Forward pass
    out_pred = model_gru(x, lengths)
    assert isinstance(out_pred, torch.Tensor), 'model output is not a torch Tensor'
    assert out_pred.shape == (batch_size, 1), 'model output shape incorrect'
    
    # Output values: the model returns raw logits (the sigmoid is applied inside
    # BCEWithLogitsLoss during training and via torch.sigmoid at inference time),
    # so we only require the outputs to be finite real numbers here.
    array_pred = out_pred.detach().numpy()
    assert np.all(np.isfinite(array_pred)), 'model output contains non-finite values'
    
    # Backward pass
    torch.mean(out_pred).backward()
    assert x.grad is not None, 'model backward gradient calculation not working'
    
# NOTE: The hidden tests perform the same kind of tests but for a larger number of 
#       randomly generated input tensors.
    

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ID: test-8-8-lstm-model - possible points: 2

input_dim = 3
hidden_dim = 64
model_lstm = LstmModel(input_dim, hidden_dim)

# At first we check if there even are any LSTM modules used in the model
assert any([isinstance(child, nn.LSTM) for child in model_lstm.children()]), (
    'The LstmModel class likely does not implement torch LSTM layer'
)
# Then we make sure that the hidden size for those LSTM modules has been 
# set correctly.
for child in model_lstm.children():
    if isinstance(child, nn.LSTM):
        assert child.hidden_size == hidden_dim, 'LSTM layer hidden size incorrect'

# The general idea of the functional tests here is that we create random input 
# vectors, perform a model forward pass and see if the model output has the correct 
# shape, consists of finite logit values, and whether or not it is possible to compute a 
# proper backward pass / gradient calculation for the given model implementation.
for _ in range(3):
    
    batch_size = np.random.randint(2, 10)
    max_length = np.random.randint(10, 20)
    
    x = torch.tensor(
        np.random.rand(batch_size, max_length, input_dim),
        dtype=torch.float32,
        requires_grad=True,
    )
    lengths = [random.randint(2, max_length) for _ in range(batch_size)]
    
    # Forward pass
    out_pred = model_lstm(x, lengths)
    assert isinstance(out_pred, torch.Tensor), 'model output is not a torch Tensor'
    assert out_pred.shape == (batch_size, 1), 'model output shape incorrect'
    
    # Output values: the model returns raw logits (the sigmoid is applied inside
    # BCEWithLogitsLoss during training and via torch.sigmoid at inference time),
    # so we only require the outputs to be finite real numbers here.
    array_pred = out_pred.detach().numpy()
    assert np.all(np.isfinite(array_pred)), 'model output contains non-finite values'
    
    # Backward pass
    torch.mean(out_pred).backward()
    assert x.grad is not None, 'model backward gradient calculation not working'
    
# NOTE: The hidden tests perform the same kind of tests but for a larger number of 
#       randomly generated input tensors.
    

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
def train_model(model: nn.Module,
                dataset: list[dict],
                epochs: int = 5,
                batch_size: int = 64,
                learning_rate: float = 1e-3,
                ) -> list[float]:
    """
    Trains the given ``model`` using the given ``dataset`` of sequential data. The model 
    will be trained for the given number of ``epochs`` using the given ``batch_size`` 
    and ``learning_rate``.
    
    The dataset should be a list containing dictionary objects where each dict represents 
    one element of the dataset. Each dict should have the following keys:
    - signal: A list representation of the ECG signal
    - target: The integer target value annotation
    
    The function returns a list of float values which represents the history of losses over the 
    various training epochs.
    """
    indices = list(range(len(dataset)))
    
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    losses_epochs = []
    
    print('starting model training...')
    for epoch in range(epochs):
        
        indices_epoch = copy(indices)
        random.shuffle(indices_epoch)
        
        losses: int = []
        index_epoch = 0
        while index_epoch < len(indices) - batch_size:
            
            indices_batch = indices_epoch[index_epoch:index_epoch+batch_size]
            data_batch = [dataset[index] for index in indices_batch]
            sequences_batch = [data['signal'] for data in data_batch]
            target_batch = [data['target'] for data in data_batch]
            
            x_batch, lengths_batch = batch_sequences(sequences_batch)
            out_pred = model(x_batch, lengths_batch)
            out_true = torch.Tensor(target_batch).unsqueeze(-1)
            
            model.zero_grad()
            loss = criterion(out_pred, out_true)
            losses.append(loss.item())
            
            loss.backward()
            optimizer.step()
    
            index_epoch += batch_size
        
        print(f' * epoch ({epoch+1}/{epochs}) - loss: {np.mean(losses):.3f}')
        losses_epochs.append(np.mean(losses))
        
    return losses_epochs


def evaluate_model(model: nn.Module,
                   sequences: list[list],
                   ) -> list[float]:
    """
    evaluates the given ``model`` predictions on the given set of ``sequences``
    
    The function returns a list of the model predictions for each sequence in the 
    given ``sequences`` list, which will be float values in the range [0, 1]
    """
    out_pred = []
    for seq in sequences:
        x, lengths = batch_sequences([seq])
        out = model(x, lengths)
        out = torch.sigmoid(out)  # logits -> [0, 1] probabilities for evaluation
        out_pred.append(out.detach().numpy()[0])
        
    return out_pred

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ~ Setting Up Dataset 

dataset = load_dataset(zip_path)
random.shuffle(dataset)

dataset_test = dataset[:500]
dataset_train = dataset[500:]

out_true = [data['target'] for data in dataset_test]

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ~ Training GRU Model

model_gru = GruModel(1, 64)

train_model(
    model=model_gru,
    dataset=dataset_train,
    epochs=5,
)
out_gru = evaluate_model(
    model=model_gru,
    sequences=[data['signal'] for data in dataset_test]
)
acc_gru = accuracy_score(out_true, np.round(out_gru))
print(f'GRU Model - test accuracy: {acc_gru*100:.2f}%')

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ID: test-8-8-gru-accuracy - possible points: 1

assert len(dataset) == 4766
assert model_gru is not None
assert isinstance(model_gru, GruModel)

assert acc_gru > 0.9

# NOTE: The hidden tests will evaluate the model on an unseen test set and check 
#       for a minimum performance, which the model will definitely achieve if it 
#       fulfills the accuracy requirement above as well.


##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ~ Training LSTM Model

model_lstm = LstmModel(1, 64)
train_model(
    model=model_lstm,
    dataset=dataset_train,
    epochs=5,
)
out_lstm = evaluate_model(
    model=model_lstm,
    sequences=[data['signal'] for data in dataset_test],
)
acc_lstm = accuracy_score(out_true, np.round(out_lstm))
print(f'LSTM Model - test accuracy: {acc_lstm*100:.2f}%')

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ID: test-8-8-lstm-accuracy - possible points: 1

assert len(dataset) == 4766
assert model_lstm is not None
assert isinstance(model_lstm, LstmModel)

assert acc_lstm > 0.9

# NOTE: The hidden tests will evaluate the model on an unseen test set and check 
#       for a minimum performance, which the model will definitely achieve if it 
#       fulfills the accuracy requirement above as well.


##### DO NOT CHANGE #####

## 8.5 Sentiment Analysis

**Free-Form Task.** In this final section, you'll have the chance to freely apply everything you've learned so far. The only goal of this section is to create a model that solves the sentiment classification problem described below. While it is recommended to apply your knowledge of recurrent neural networks to solve this task, in principle you are free to use any method you want, so long as it solves the given problem.

**Sentiment Classification.** In this section, we will apply the concept of recurrent neural networks to the task of sentiment classification. For a given piece of text, the goal is to classify the underlying sentiment as either *negative* or *positive*. To illustrate this, consider the following simple examples: 

- I really liked this cake. (Positive)
- I've had a terrible day! (Negative)

Based on these, one might think that such a task mainly comes down to detecting the presence of certain negative and positive indicators (good, bad, ...). While this may be true for the majority of samples in practice, one also has to consider edge cases such as negations and sarcasm:

- This restaurant really isn't bad! (Positive)
- That meeting was sooo necessary... I am very glad it happened (Negative)

**Tweet Sentiment Dataset.** The specific dataset we'll be using consists of 10k short [tweets](https://en.wikipedia.org/wiki/Twitter) in which users give their opinion concerning a variety of different video games. Each tweet is annotated with either the *Positive* or the *Negative* label, indicating the general sentiment of its content. The following are some examples from the dataset:

- RhandlerR RhandlerR RhandlerR RhandlerR . This doesn’t look right. At all. Why am I just noticing this now? Shouldn’t the payouts be better in M4? Or are these typos. (Negative)
- Finally made it into the base 10, just to try to hold on to it. Feels like enough really good meta right now in pirate pirates with all of the odd dh to even mage in high legend... (Positive)


<div style="border: 1px solid #CEB037; border-radius: 3px; padding: 6px; background-color: #faf7e0ff; color: black;">
<strong>🛠️ Task 8.9 (7 points).</strong> In this task, the overall goal is to solve the previously described sentiment classification problem for the Twitter dataset. While it is recommended to do so using an RNN model, you are open to use any method you want, so long as your model class implements the <code>SentimentInterface</code> that is introduced below. To solve the exercise simply assign a new instance of your custom model implementation to the <code>model</code> variable in the cells below.
<br><br>
<strong>⚠️ Important — read before you start.</strong> When your notebook is graded it runs <em>without internet access</em> and with a <em>per-cell time limit of about 5 minutes</em>, so you <strong>cannot train a competitive model from scratch during grading</strong>. The intended workflow is: (1) train your model <em>offline</em> in your own session using the cells below; (2) save the trained weights and upload them to the university's <strong>BwSync&amp;Share</strong> cloud, which is the one external host still reachable at grading time (use the <code>nextcloud_download</code> helper, as elsewhere in this notebook); and (3) in the final cell, simply <em>download and load</em> those weights instead of training, so your <code>model</code> is ready well within the time limit.
<br><br>
The points for this exercise will be distributed incrementally based on the performance of your model.
<ul>
<li><code>+1 points</code>: If your model class confirms to the given interface and is able to produce proper predictions (only considers general functionality, not yet the correctness of the predictions!)</li>
<li><code>+2 points</code>: If your model achieves a performance which is better than random guessing (&gt;60% accuracy)</li>
<li><code>+2 points</code>: If your model achieves <em>decent</em> performance (&gt;75% accuracy)</li>
<li><code>+2 points</code>: If your model achieves <em>good</em> performance (&gt;90% accuracy)</li>
</ul>
The performance will be evaluated on an unseen test dataset. This test dataset consists of 100 elements and is balanced regarding the output labels (same number of negative and positive samples).
</div>

In [ ]:
##### DO NOT CHANGE #####
import typing as typ

class SentimentInterface:
    """
    This interface has to be implemented for a sentiment classification model which 
    solves the given task. To implement this interface for a custom model class, simply 
    inherit from it and overwrite the ``predict_sentiment`` method with your custom 
    solution.
    """
    
    def predict_sentiment(self, text: str) -> typ.Literal['Negative', 'Positive']:
        """
        This function receives a single string ``text`` and is supposed to implement 
        some sort of sentiment classification method to finally return either of the 
        string labels "Negative" or "Positive" to determine the given text's overall 
        sentiment.
        
        :param text: the input text string
        
        :returns: String literal which is either "Negative" in case of a negative 
            sentiment and "Positive" in case of a positive sentiment.
        """
        raise NotImplementedError
        
        
# ~ Mock Implementation of a naive sentiment classification approach

class MockModel(SentimentInterface):
    
    def predict_sentiment(self, text: str) -> str:
        if 'good' in text:
            return 'Positive'
        else:
            return 'Negative'

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# The dataset is downloaded from the file share server and stored as a DataFrame
content = nextcloud_download('https://bwsyncandshare.kit.edu/s/nAy6oMPdP7cKAz7')
df: pd.DataFrame = pd.read_csv(io.StringIO(content))

##### DO NOT CHANGE #####

In [ ]:
# == DATA EXPLORATION ==
# You may use this cell to perform some exploratory data analysis in order to get 
# a better overview of what type of dataset you'll be working with.

# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# == MODEL IMPLEMENTATION ==
# You may use this cell to define your model implementation.
# In case you are planning to implement an RNN solution, here are several pointers of
# what aspects to consider:
# - The first challenge is how to convert the text itself into a sequence which a
#   neural network model can understand. You'll have to find some way to meaningfully
#   convert the string into a series of numeric vectors.
#   For this task, several solutions of varying complexity exist. A simple solution might
#   be a one-hot encoding for characters and more complex solutions can use third-party
#   "tokenizers".

# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# === MODEL TRAINING ===
# You may use this cell to train your model.
# For your training, keep in mind the cell execution timeout during grading! If a 
# cell runs longer than 5 minutes it will automatically be terminated, so make 
# sure to set up your training accordingly!

# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# === MODEL EVALUATION ===
# You may use this cell to evaluate the performance of your model. 
# To do this, you might want to think about splitting the dataset that is available to you 
# into appropriate train and validation sets. Your model's performance on an unseen 
# validation split should give you a better indication of the final performance on 
# the unseen test set.

# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
# TASK: Create a new instance of your custom model implementation which inherits
#       from the "SentimentInterface" and assign this instance to the model
#       variable below.

# HINT: During the grading of the notebook, general access to the internet is blocked.
#       This means that it won't be possible to simply send the text snippets to
#       a ChatGPT and or HuggingFace API to solve the task for you!

# HINT: However, access to the university's BwSync&Share cloud IS enabled. Since training
#       a model from scratch here would exceed the per-cell timeout, the intended approach is
#       to train your model offline and load its pre-trained weights from your cloud storage.

model: SentimentInterface = None

# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: test-8-9-functionality - possible points: 1

assert model is not None
assert isinstance(model, SentimentInterface), 'make sure the model implements the interface!'

reviews_ = [
    'This is the worst game I have ever played in my life!',
    'The GREATEST experience of all time - I am so happy',
]
for review in reviews_:
    prediction = model.predict_sentiment(review)
    print(f'input: "{review}" - prediction: {prediction}')
    assert isinstance(prediction, str)
    assert prediction in ['Negative', 'Positive']

# NOTE: There are no additional hidden test cases in this cell
    

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ID: test-8-9-basic-acc - possible points: 2

# NOTE: The hidden tests will evaluate the model on the unseen test set and check 
#       if the accuracy is >= 0.60


##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ID: test-8-9-decent-acc - possible points: 2

# NOTE: The hidden tests will evaluate the model on the unseen test set and check 
#       if the accuracy is >= 0.75


##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ID: test-8-9-good-acc - possible points: 2

# NOTE: The hidden tests will evaluate the model on the unseen test set and check 
#       if the accuracy is >= 0.90


##### DO NOT CHANGE #####

👋 I hope you've found this exercise sheet helpful for your understanding of recurrent neural networks! The final free-form task was a bit of an experiment that deviated slightly from the usually more guided style of the exercises.